# Simulación Computacional: Atomos Frios y Optica Cuantica
**Área:** Fisica Atomica y Molecular

Este cuaderno interactivo de Jupyter ejecuta numéricamente los modelos físicos descritos en el repositorio. Puedes modificar los parámetros iniciales y ejecutar las celdas para visualizar gráficos o animaciones interactivos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Parámetros del grid y de simulación
N = 512
x_max = 10.0
x = np.linspace(-x_max, x_max, N)
dx = x[1] - x[0]
dt = 0.005  # Paso de tiempo imaginario
steps = 2000

# Potencial Armónico de la Trampa Magnética
V = 0.5 * x**2

# Fuerza de interacción entre átomos (parámetro no lineal g)
# g > 0 implica repulsión (ensanchamiento del condensado)
g_values = [0.0, 10.0, 50.0]

plt.figure(figsize=(10, 6))
plt.plot(x, V, 'k--', label='Potencial de Trampa $V(x)$')

for g in g_values:
    # Estado inicial gaussiano de prueba
    psi = np.exp(-0.5 * x**2)
    psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx)
    
    # Evolución en tiempo imaginario (psi -> exp(-H*tau) psi)
    # Utilizamos el método Split-Step Fourier
    k = np.fft.fftfreq(N, d=dx) * 2 * np.pi
    T_op = np.exp(-0.5 * k**2 * dt)
    
    for _ in range(steps):
        # Medio paso espacial (Potencial + Interacción no lineal)
        V_eff = V + g * np.abs(psi)**2
        psi *= np.exp(-0.5 * V_eff * dt)
        
        # Paso en el espacio de momentos (Energía cinética)
        psi = np.fft.fft(psi)
        psi *= T_op
        psi = np.fft.ifft(psi)
        
        # Medio paso espacial restante
        V_eff = V + g * np.abs(psi)**2
        psi *= np.exp(-0.5 * V_eff * dt)
        
        # Renormalización obligatoria en tiempo imaginario
        psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx)
        
    densidad = np.abs(psi)**2
    plt.plot(x, densidad, lw=2, label=f'BEC Densidad (g={g})')

plt.xlim(-6, 6)
plt.ylim(0, 1.2)
plt.xlabel('Posición x')
plt.ylabel('Densidad $|\psi(x)|^2$')
plt.title('Estado Fundamental de un BEC (Ecuación de Gross-Pitaevskii)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
# plt.show()